In [1]:
#importing libraries
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score,
    recall_score,
    matthews_corrcoef,
    make_scorer
)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

In [4]:
#loading dataset
df = pd.read_csv("yeast.csv") 
df.head(),df.shape

(    mcg   gvh   alm   mit  erl  pox   vac   nuc name
 0  0.58  0.61  0.47  0.13  0.5  0.0  0.48  0.22  MIT
 1  0.43  0.67  0.48  0.27  0.5  0.0  0.53  0.22  MIT
 2  0.64  0.62  0.49  0.15  0.5  0.0  0.53  0.22  MIT
 3  0.58  0.44  0.57  0.13  0.5  0.0  0.54  0.22  NUC
 4  0.42  0.44  0.48  0.54  0.5  0.0  0.48  0.22  MIT,
 (1484, 9))

In [5]:
#EDA
df.info()
df.describe()
df["name"].value_counts()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1484 entries, 0 to 1483
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   mcg     1484 non-null   float64
 1   gvh     1484 non-null   float64
 2   alm     1484 non-null   float64
 3   mit     1484 non-null   float64
 4   erl     1484 non-null   float64
 5   pox     1484 non-null   float64
 6   vac     1484 non-null   float64
 7   nuc     1484 non-null   float64
 8   name    1484 non-null   object 
dtypes: float64(8), object(1)
memory usage: 104.5+ KB


mcg     0
gvh     0
alm     0
mit     0
erl     0
pox     0
vac     0
nuc     0
name    0
dtype: int64

In [6]:
#separating features and target
X = df.drop(columns=["name"])
y = df["name"]

In [11]:
#Encoding target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [7]:
#re-checking feature types
X.dtypes

mcg    float64
gvh    float64
alm    float64
mit    float64
erl    float64
pox    float64
vac    float64
nuc    float64
dtype: object

In [8]:
#decision tree
dt = DecisionTreeClassifier(
    criterion="entropy",   
    random_state=42
)


In [9]:
#cross validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [12]:
# Baseline Decision Tree Evaluation

f1 = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(f1_score, average="macro")
)

recall = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(recall_score, average="macro")
)

mcc = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring=make_scorer(matthews_corrcoef)
)

auc = cross_val_score(
    dt, X, y_encoded,
    cv=skf,
    scoring="roc_auc_ovr"
)

print("Decision Tree (Baseline)")
print("F1 (macro):", f1.mean())
print("Recall (macro):", recall.mean())
print("MCC:", mcc.mean())
print("AUC:", auc.mean())


Decision Tree (Baseline)
F1 (macro): 0.40109448185954044
Recall (macro): 0.40686597645768485
MCC: 0.36260526129215565
AUC: 0.6711578806889331


In [13]:
#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
param_grid = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

grid = GridSearchCV(
    DecisionTreeClassifier(criterion="entropy", random_state=42),
    param_grid,
    cv=skf,
    scoring="f1_macro"
)

grid.fit(X, y_encoded)

print("Best parameters:", grid.best_params_)

Best parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}


In [14]:
best_dt = grid.best_estimator_


In [16]:
# Random Forest 

best_rf = grid.best_estimator_

f1_rf = cross_val_score(
    best_rf, X, y_encoded,
    cv=skf,
    scoring=make_scorer(f1_score, average="macro")
)

recall_rf = cross_val_score(
    best_rf, X, y_encoded,
    cv=skf,
    scoring=make_scorer(recall_score, average="macro")
)

mcc_rf = cross_val_score(
    best_rf, X, y_encoded,
    cv=skf,
    scoring=make_scorer(matthews_corrcoef)
)

auc_rf = cross_val_score(
    best_rf, X, y_encoded,
    cv=skf,
    scoring="roc_auc_ovr"
)

print("Random Forest (Tuned)")
print("F1 (macro):", f1_rf.mean())
print("Recall (macro):", recall_rf.mean())
print("MCC:", mcc_rf.mean())
print("AUC:", auc_rf.mean())


Random Forest (Tuned)
F1 (macro): 0.42956744740891584
Recall (macro): 0.44185439391304476
MCC: 0.44313859571382264
AUC: 0.7513039460902352


In [17]:
# AdaBoost 

ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

f1_ada = cross_val_score(
    ada, X, y_encoded,
    cv=skf,
    scoring=make_scorer(f1_score, average="macro")
)

recall_ada = cross_val_score(
    ada, X, y_encoded,
    cv=skf,
    scoring=make_scorer(recall_score, average="macro")
)

mcc_ada = cross_val_score(
    ada, X, y_encoded,
    cv=skf,
    scoring=make_scorer(matthews_corrcoef)
)

auc_ada = cross_val_score(
    ada, X, y_encoded,
    cv=skf,
    scoring="roc_auc_ovr"
)

print("AdaBoost")
print("F1 (macro):", f1_ada.mean())
print("Recall (macro):", recall_ada.mean())
print("MCC:", mcc_ada.mean())
print("AUC:", auc_ada.mean())


AdaBoost
F1 (macro): 0.33695306620244775
Recall (macro): 0.34916410930320335
MCC: 0.3566009915673077
AUC: 0.8220468545183355


In [21]:
# Comparison Table

results = pd.DataFrame({
    "Model": [
        "Decision Tree (Baseline)",
        "Decision Tree (Tuned)",
        "Random Forest (Tuned)",
        "AdaBoost"
    ],
    "F1 (macro)": [
        f1.mean(),
        f1.mean(),
        f1_rf.mean(),
        f1_ada.mean()
    ],
    "Recall (macro)": [
        recall.mean(),
        recall.mean(),
        recall_rf.mean(),
        recall_ada.mean()
    ],
    "MCC": [
        mcc.mean(),
        mcc.mean(),
        mcc_rf.mean(),
        mcc_ada.mean()
    ],
    "AUC": [
        auc.mean(),
        auc.mean(),
        auc_rf.mean(),
        auc_ada.mean()
    ]
})

results


,Model,F1 (macro),Recall (macro),MCC,AUC
0,Decision Tree (Baseline),0.401094,0.406866,0.362605,0.671158
1,Decision Tree (Tuned),0.401094,0.406866,0.362605,0.671158
2,Random Forest (Tuned),0.429567,0.441854,0.443139,0.751304
3,AdaBoost,0.336953,0.349164,0.356601,0.822047
